# Permute predictions by randomly swapping predictions

- For alternative hypothesis: 2T - 3Q > 0
    - Text adds more info to raw sum predictions than questionnaire responses


## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.perm_res import expand_alpha_rows
import pandas as pd
import numpy as np
from shutil import rmtree

PERM_LOOP_DIR = BASE_PATH / "perm_res/preds"
perm_map_export_dir = BASE_PATH / "perm_res/switch/map"
N_PERMS = 1000

## Construct reference df for permutations

- Each column pertains to one of n permutations
- Each row pertains to a single participant
- Values will be 1/-1

In [ ]:
# Use arbitrary base data to get DAWBA_ID cols for parent & self
parent_ids = pd.read_parquet(
    BASE_PATH / "data/w_probs/Llama-3.3-70B-Instruct/pt_noshort/parent.parquet"
)["DAWBA_ID"]
self_ids = pd.read_parquet(
    BASE_PATH / "data/w_probs/Llama-3.3-70B-Instruct/pt_noshort/self.parquet"
)["DAWBA_ID"]

In [ ]:
def get_perm_df(id_list, loop_dir):
    df = id_list.reset_index(drop=True)
    rand_sequences = []
    for perm_dir in loop_dir.iterdir():
        perm_name = perm_dir.name
        if perm_name == "baseline":
            continue
        with open(perm_dir / "seed.txt") as f:
            seed_for_perm = int(f.read())
        rng = np.random.default_rng(seed_for_perm)
        rand_seq = rng.choice([0, 1], size=len(df))
        rand_sequences.append(pd.Series(rand_seq, name=perm_name))

    return pd.concat([df] + rand_sequences, axis=1)

In [8]:
parent_perm_map = get_perm_df(parent_ids, PERM_LOOP_DIR)
self_perm_map = get_perm_df(self_ids, PERM_LOOP_DIR)

In [9]:
if perm_map_export_dir.exists():
    rmtree(perm_map_export_dir)
perm_map_export_dir.mkdir(exist_ok=False, parents=True)
parent_perm_map.to_parquet(perm_map_export_dir / "parent.parquet")
self_perm_map.to_parquet(perm_map_export_dir / "self.parquet")

## Get df containing diffs for each participant/model/repeat

In [ ]:
def get_base_df(impact_type):
    """
    Merge predictions from 2T and 3Q
    Single row pertains to a unique participant/repeat/model_config combination
        - Also contains diff column computed as (2T-3Q)
    """
    og_df = pd.read_parquet(
        BASE_PATH / f"perm_res/preds/baseline/{impact_type}/pred_df_combined.parquet"
    )
    keep_cols = [
        "DAWBA_ID",
        "true",
        "repeat",
        "agg_type",
        "llama_model",
        "desc_seg",
        "alpha",
    ]
    og_df_2_layers = og_df[og_df["num_layers"] == 2][keep_cols + ["pred_stage3_total"]]
    og_df_3_layers = og_df[og_df["num_layers"] == 3][keep_cols + ["pred_stage2_total"]]
    # Remove unnecessary duplicated rows in 3 layer df
    #   - 3T uses second alpha val but 3Q does not
    og_df_3_layers["alpha1"] = og_df_3_layers["alpha"].str.split("_").str[0]
    og_df_3_layers_red = og_df_3_layers.drop_duplicates(
        subset=[
            "DAWBA_ID",
            "true",
            "repeat",
            "agg_type",
            "llama_model",
            "desc_seg",
            "alpha1",
            "pred_stage2_total",
        ]
    )
    og_df_3_layers_red = og_df_3_layers_red.drop(["alpha"], axis=1)
    og_df_3_layers_red.rename(columns={"alpha1": "alpha"}, inplace=True)
    ## MERGE
    df_merged_layers = og_df_2_layers.merge(
        og_df_3_layers_red,
        on=[
            "DAWBA_ID",
            "true",
            "repeat",
            "agg_type",
            "llama_model",
            "desc_seg",
            "alpha",
        ],
    )
    df_merged_layers = df_merged_layers.rename(
        columns={
            "pred_stage3_total": "pred_2T",
            "pred_stage2_total": "pred_3Q",
        }
    )
    # Ensure expected length
    unique_model_configs = len(
        df_merged_layers.drop_duplicates(
            [
                "agg_type",
                "llama_model",
                "desc_seg",
                "alpha",
            ]
        )
    )
    num_repeats = 10
    unique_ids = df_merged_layers["DAWBA_ID"].nunique()
    assert unique_model_configs * num_repeats * unique_ids == len(df_merged_layers)
    return df_merged_layers

In [3]:
parent_base_df = get_base_df("parent")
self_base_df = get_base_df("self")

## Permute (switch) base_df diffs based on perm_maps

In [ ]:
parent_perm_map = pd.read_parquet(perm_map_export_dir / "parent.parquet")
self_perm_map = pd.read_parquet(perm_map_export_dir / "self.parquet")

In [ ]:
def perm_by_switching(base_df, map_df):
    base_map_merge = base_df.merge(map_df, on="DAWBA_ID")
    cols_to_drop = []
    new_cols = {}
    for perm_col in map_df.columns:
        if perm_col == "DAWBA_ID":
            continue
        mask = base_map_merge[perm_col]
        new_cols[f"pred_2T_{perm_col}"] = np.where(
            mask, base_map_merge["pred_3Q"], base_map_merge["pred_2T"]
        )
        new_cols[f"pred_3Q_{perm_col}"] = np.where(
            mask, base_map_merge["pred_2T"], base_map_merge["pred_3Q"]
        )
        cols_to_drop.append(perm_col)
    base_map_merge = pd.concat(
        [
            base_map_merge.drop(columns=cols_to_drop),
            pd.DataFrame(new_cols, index=base_map_merge.index),
        ],
        axis=1,
    )
    return base_map_merge

In [29]:
parent_permed = perm_by_switching(parent_base_df, parent_perm_map)
self_permed = perm_by_switching(self_base_df, self_perm_map)

In [31]:
permed_df_export_dir = BASE_PATH / "perm_res/switch/perm_diffs"
if permed_df_export_dir.exists():
    rmtree(permed_df_export_dir)
permed_df_export_dir.mkdir(exist_ok=False, parents=True)
parent_permed.to_parquet(permed_df_export_dir / "parent.parquet")
self_permed.to_parquet(permed_df_export_dir / "self.parquet")